In [16]:
from pathlib import Path
DATASETS_DIR = Path('datasets')

In [17]:
J_D = DATASETS_DIR / 'data-from-juniors'/'labels_final.xlsx'

In [18]:
import pandas as pd 


In [19]:
df = pd.read_excel(J_D)

In [21]:
df.columns

Index(['video_id', 'generic', 'humour', 'positive', 'sensitive',
       'derogatory__lang', 'threat', 'sexuality_hate', 'nationality_hate',
       'caste_based_hate', 'political_hate', 'religion_hate', 'informative',
       'ethinity_hate', 'anger', 'emotional', 'social_hate', 'controversial',
       'indv_hate', 'gender_hate'],
      dtype='str')

In [24]:
!pip install opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ------ --------------------------------- 6.6/40.2 MB 36.6 MB/s eta 0:00:01
   ----------- ---------------------------- 11.8/40.2 MB 30.8 MB/s eta 0:00:01
   ------------ --------------------------- 12.8/40.2 MB 23.0 MB/s eta 0:00:02
   ------------- -------------------------- 13.4/40.2 MB 17.9 MB/s eta 0:00:02
   ------------- -------------------------- 13.9/40.2 MB 14.5 MB/s eta 0:00:02
   -------------- ------------------------- 14.9/40.2 MB 12.5 MB/s eta 0:00:03
   ---------------- ----------------------- 16.3/40.2 MB 11.6 MB/s eta 0:00:03
   ------------------ --------------------- 18.4/40.2 MB 11.3 MB/s eta 0:00:02
   -------------------- ------------------- 20.7/40.2 MB 11.3 MB/s eta 0:00:02
   ----------------------- ---------------- 23.3/40.2 MB 11.5 MB/s eta 0:00:02
   -------------------------- ------------- 26.2/40.2 MB 11.8 MB/s eta 0:00:02
   ----------------------------- ---------- 29.6/40.2 MB 12.1 

In [25]:
import os
import cv2
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VIDEO_DIR = "./datasets/data-from-juniors/videos"
CSV_PATH = "dataset.csv"

NUM_FRAMES = 8
IMG_SIZE = 224

In [28]:
df = pd.read_excel(J_D)

LABEL_COLUMNS = df.columns[1:]  # all except video_id
NUM_LABELS = len(LABEL_COLUMNS)

print("Labels:", LABEL_COLUMNS)

Labels: Index(['generic', 'humour', 'positive', 'sensitive', 'derogatory__lang',
       'threat', 'sexuality_hate', 'nationality_hate', 'caste_based_hate',
       'political_hate', 'religion_hate', 'informative', 'ethinity_hate',
       'anger', 'emotional', 'social_hate', 'controversial', 'indv_hate',
       'gender_hate'],
      dtype='str')


In [29]:
LABEL_COLUMNS = df.columns[1:]

print("Total samples:", len(df))
print("Total labels:", len(LABEL_COLUMNS))

# -----------------------
# 1. PER-LABEL DISTRIBUTION
# -----------------------
label_counts = df[LABEL_COLUMNS].sum().sort_values(ascending=False)

print("\n=== LABEL DISTRIBUTION ===")
print(label_counts)

print("\n=== LABEL PERCENTAGE ===")
print((label_counts / len(df)) * 100)

# -----------------------
# 2. MULTI-LABEL COUNT PER SAMPLE
# -----------------------
df["num_labels"] = df[LABEL_COLUMNS].sum(axis=1)

print("\n=== LABELS PER VIDEO ===")
print(df["num_labels"].describe())

print("\nDistribution:")
print(df["num_labels"].value_counts().sort_index())

# -----------------------
# 3. HOW MANY VIDEOS HAVE NO LABELS
# -----------------------
no_label = (df["num_labels"] == 0).sum()
print(f"\nVideos with NO labels: {no_label}")

# -----------------------
# 4. CO-OCCURRENCE MATRIX
# -----------------------
co_matrix = df[LABEL_COLUMNS].T.dot(df[LABEL_COLUMNS])

print("\n=== CO-OCCURRENCE (sample) ===")
print(co_matrix.head())

# -----------------------
# 5. TOP CO-OCCURRING PAIRS
# -----------------------
pairs = []

for i in range(len(LABEL_COLUMNS)):
    for j in range(i + 1, len(LABEL_COLUMNS)):
        l1 = LABEL_COLUMNS[i]
        l2 = LABEL_COLUMNS[j]
        count = co_matrix.loc[l1, l2]
        pairs.append((l1, l2, count))

pairs = sorted(pairs, key=lambda x: x[2], reverse=True)

print("\n=== TOP 10 CO-OCCURRING LABELS ===")
for p in pairs[:10]:
    print(p)

Total samples: 5153
Total labels: 19

=== LABEL DISTRIBUTION ===
humour              4415
sensitive            702
anger                373
derogatory__lang     351
generic              318
positive             190
emotional            165
threat               150
political_hate       131
gender_hate          109
indv_hate             68
nationality_hate      63
informative           58
sexuality_hate        39
controversial         31
religion_hate         29
social_hate           24
caste_based_hate      13
ethinity_hate         10
dtype: int64

=== LABEL PERCENTAGE ===
humour              85.678246
sensitive           13.623132
anger                7.238502
derogatory__lang     6.811566
generic              6.171162
positive             3.687173
emotional            3.202018
threat               2.910926
political_hate       2.542208
gender_hate          2.115273
indv_hate            1.319620
nationality_hate     1.222589
informative          1.125558
sexuality_hate       0.756841
c

In [30]:
df = df.drop(columns=["humour"])

In [31]:
rare_labels = [
    "ethinity_hate",
    "caste_based_hate",
    "social_hate",
    "religion_hate",
    "controversial"
]

df["rare_hate"] = df[rare_labels].max(axis=1)
df = df.drop(columns=rare_labels)

In [32]:
LABEL_COLUMNS = df.columns[1:]

print("Total samples:", len(df))
print("Total labels:", len(LABEL_COLUMNS))

# -----------------------
# 1. PER-LABEL DISTRIBUTION
# -----------------------
label_counts = df[LABEL_COLUMNS].sum().sort_values(ascending=False)

print("\n=== LABEL DISTRIBUTION ===")
print(label_counts)

print("\n=== LABEL PERCENTAGE ===")
print((label_counts / len(df)) * 100)

# -----------------------
# 2. MULTI-LABEL COUNT PER SAMPLE
# -----------------------
df["num_labels"] = df[LABEL_COLUMNS].sum(axis=1)

print("\n=== LABELS PER VIDEO ===")
print(df["num_labels"].describe())

print("\nDistribution:")
print(df["num_labels"].value_counts().sort_index())

# -----------------------
# 3. HOW MANY VIDEOS HAVE NO LABELS
# -----------------------
no_label = (df["num_labels"] == 0).sum()
print(f"\nVideos with NO labels: {no_label}")

# -----------------------
# 4. CO-OCCURRENCE MATRIX
# -----------------------
co_matrix = df[LABEL_COLUMNS].T.dot(df[LABEL_COLUMNS])

print("\n=== CO-OCCURRENCE (sample) ===")
print(co_matrix.head())

# -----------------------
# 5. TOP CO-OCCURRING PAIRS
# -----------------------
pairs = []

for i in range(len(LABEL_COLUMNS)):
    for j in range(i + 1, len(LABEL_COLUMNS)):
        l1 = LABEL_COLUMNS[i]
        l2 = LABEL_COLUMNS[j]
        count = co_matrix.loc[l1, l2]
        pairs.append((l1, l2, count))

pairs = sorted(pairs, key=lambda x: x[2], reverse=True)

print("\n=== TOP 10 CO-OCCURRING LABELS ===")
for p in pairs[:10]:
    print(p)

Total samples: 5153
Total labels: 15

=== LABEL DISTRIBUTION ===
num_labels          7239
sensitive            702
anger                373
derogatory__lang     351
generic              318
positive             190
emotional            165
threat               150
political_hate       131
gender_hate          109
rare_hate             78
indv_hate             68
nationality_hate      63
informative           58
sexuality_hate        39
dtype: int64

=== LABEL PERCENTAGE ===
num_labels          140.481273
sensitive            13.623132
anger                 7.238502
derogatory__lang      6.811566
generic               6.171162
positive              3.687173
emotional             3.202018
threat                2.910926
political_hate        2.542208
gender_hate           2.115273
rare_hate             1.513681
indv_hate             1.319620
nationality_hate      1.222589
informative           1.125558
sexuality_hate        0.756841
dtype: float64

=== LABELS PER VIDEO ===
count    5153.0

In [33]:
drop_cols = ["generic", "positive", "informative"]
df = df.drop(columns=drop_cols)

In [34]:
df = df.drop(columns=["num_labels"])

In [35]:
LABEL_COLUMNS = [col for col in df.columns if col != "video_id"]

In [36]:
label_counts = df[LABEL_COLUMNS].sum()
total = len(df)

pos_weight = (total - label_counts) / label_counts
pos_weight = torch.tensor(pos_weight.values, dtype=torch.float).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [37]:
pos_weight = torch.clamp(pos_weight, max=20.0)

In [38]:
LABEL_COLUMNS

['sensitive',
 'derogatory__lang',
 'threat',
 'sexuality_hate',
 'nationality_hate',
 'political_hate',
 'anger',
 'emotional',
 'indv_hate',
 'gender_hate',
 'rare_hate']